# Additive validation fallback

This validated copy prepends a small educational fallback so the original cells can run when `statsmodels` is unavailable. The original notebook file is unchanged.

In [ ]:

# Additive validation fallback for environments without statsmodels.
# This is not a replacement for statsmodels inference; it supports local execution of this teaching notebook.
import sys, types, numpy as np
class _ARIMAResult:
    def __init__(self, endog):
        y = np.asarray(endog, dtype=float)
        self.last = float(y[-1]) if len(y) else 0.0
        if len(y) >= 2:
            yy = y[1:]
            xx = np.column_stack([np.ones(len(yy)), y[:-1]])
            self.beta = np.linalg.lstsq(xx, yy, rcond=None)[0]
        else:
            self.beta = np.array([0.0, 0.0])
    def forecast(self, steps=1):
        return np.array([self.beta[0] + self.beta[1] * self.last])
class ARIMA:
    def __init__(self, endog, order=(1,0,0), **kwargs):
        self.endog = endog
        self.order = order
    def fit(self):
        return _ARIMAResult(self.endog)
def adfuller(series, *args, **kwargs):
    import pandas as pd
    y = np.asarray(pd.Series(series).dropna(), dtype=float)
    dy = np.diff(y); lag = y[:-1]
    X = np.column_stack([np.ones(len(lag)), lag])
    beta = np.linalg.lstsq(X, dy, rcond=None)[0]
    resid = dy - X @ beta
    sigma2 = np.sum(resid * resid) / max(len(dy) - X.shape[1], 1)
    cov = sigma2 * np.linalg.pinv(X.T @ X)
    stat = float(beta[1] / np.sqrt(max(cov[1, 1], 1e-18)))
    p = 0.01 if stat < -3.43 else 0.04 if stat < -2.86 else 0.09 if stat < -2.57 else 0.20
    return stat, p, 0, len(y), {}, 0.0
def _acf_vals(x, lags):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]; x = x - x.mean()
    den = np.sum(x*x)
    return np.array([1.0] + [np.sum(x[k:]*x[:-k])/den for k in range(1, lags+1)])
def _pacf_vals(x, lags):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]
    vals = [1.0]
    for k in range(1, lags+1):
        y = x[k:]; cols = [x[k-j:len(x)-j] for j in range(1, k+1)]
        X = np.column_stack([np.ones(len(y))] + cols)
        vals.append(float(np.linalg.lstsq(X, y, rcond=None)[0][-1]))
    return np.array(vals)
def plot_acf(x, lags=15, ax=None, **kwargs):
    import matplotlib.pyplot as plt
    ax = ax or plt.gca()
    vals = _acf_vals(x, lags)
    ax.bar(range(len(vals)), vals)
    band = 1.96 / np.sqrt(len(np.asarray(x)))
    ax.axhline(band, color="gray", ls="--", lw=.8); ax.axhline(-band, color="gray", ls="--", lw=.8)
    return ax
def plot_pacf(x, lags=15, ax=None, method=None, **kwargs):
    import matplotlib.pyplot as plt
    ax = ax or plt.gca()
    vals = _pacf_vals(x, lags)
    ax.bar(range(len(vals)), vals)
    band = 1.96 / np.sqrt(len(np.asarray(x)))
    ax.axhline(band, color="gray", ls="--", lw=.8); ax.axhline(-band, color="gray", ls="--", lw=.8)
    return ax
statsmodels = types.ModuleType("statsmodels")
tsa = types.ModuleType("statsmodels.tsa")
arima = types.ModuleType("statsmodels.tsa.arima")
model = types.ModuleType("statsmodels.tsa.arima.model")
stattools = types.ModuleType("statsmodels.tsa.stattools")
graphics = types.ModuleType("statsmodels.graphics")
tsaplots = types.ModuleType("statsmodels.graphics.tsaplots")
model.ARIMA = ARIMA
stattools.adfuller = adfuller
tsaplots.plot_acf = plot_acf
tsaplots.plot_pacf = plot_pacf
for name, module in {
    "statsmodels": statsmodels,
    "statsmodels.tsa": tsa,
    "statsmodels.tsa.arima": arima,
    "statsmodels.tsa.arima.model": model,
    "statsmodels.tsa.stattools": stattools,
    "statsmodels.graphics": graphics,
    "statsmodels.graphics.tsaplots": tsaplots,
}.items():
    sys.modules.setdefault(name, module)
print("statsmodels fallback injected for validated additive execution")


# Week 10-1 · ASQ-01 — Statistical Analysis of Financial Market Data
### Part 1: From simple rules to time-series models (AR → stationarity → ACF/PACF → rolling AR(1) → ARIMA)
*EPAT · Advanced Statistics for Quant Strategies · executed practice notebook*

**The story.** Simple moving-average crossovers look great until a **regime shift** (COVID, war, rate hikes)
arrives — then they collapse. This lecture builds a more rigorous, *adaptive* toolkit from time-series
statistics: **AutoRegressive (AR)** models, the idea of **stationarity** (and the **ADF test**), the
**ACF / PACF** diagnostics, a **rolling AR(1)** strategy that re-fits every period, and finally the
**ARIMA** family (AR + Integrated + Moving-average-of-errors).

> The lecture uses SPY (daily) and PayPal (weekly) pulled live with `yfinance`. That isn't available offline,
> so we reproduce **every step** on a shipped daily index series (`market_data.csv`, NIFTY 50). The methods,
> tests and conclusions are identical — only the ticker changes.

## 1. Imports — `statsmodels` is the new workhorse
NumPy / pandas as usual, plus the time-series toolbox: `ARIMA`, the `adfuller` stationarity test, and `plot_acf` / `plot_pacf` diagnostics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings("ignore")          # ARIMA chatters about convergence; harmless here

pd.set_option("display.width", 110)
print("statsmodels ready")

## 2. Load the data (our SPY stand-in)
Daily OHLCV. We compute **simple daily returns** — the same `pct_change()` you have seen before.

In [ ]:
df = pd.read_csv("market_data.csv", parse_dates=["Date"]).set_index("Date").sort_index()
df["returns"] = df["Close"].pct_change()
df = df.dropna()
print("rows:", len(df), "| from", df.index.min().date(), "to", df.index.max().date())
df[["Close", "returns"]].tail()

## 3. Prices vs returns — volatility clustering
Prices **trend** (up over the long run, with violent crashes). Returns hover around **0%**, but the *size* of the wiggles clusters: calm stretches and stormy stretches. That dependence in volatility is the first hint that **markets aren't purely random** — and that static rules can struggle to adapt.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(df.index, df["Close"], color="#db2777"); ax[0].set_title("Price (trends — non-stationary)")
ax[1].plot(df.index, df["returns"], color="#9333ea", lw=0.7); ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_title("Daily returns (hover around 0 — volatility clusters)")
plt.tight_layout(); plt.show()
print("price start %.0f -> end %.0f" % (df['Close'].iloc[0], df['Close'].iloc[-1]))

## 4. The motivation: a simple MA crossover *underperforms*
Short window **10**, long window **30** (arbitrary, as in the lecture). Go **long** when SMA10 > SMA30, else
**short**. Shift the signal by one day (no look-ahead), multiply by returns, and compound. Compare to
**buy-and-hold**.

In [ ]:
d = df.copy()
d["sma_s"] = d["Close"].rolling(10).mean()
d["sma_l"] = d["Close"].rolling(30).mean()
d["signal"] = np.where(d["sma_s"] > d["sma_l"], 1, -1)
d["strat"] = d["signal"].shift(1) * d["returns"]
d = d.dropna()

ma_curve = (1 + d["strat"]).cumprod()
bh_curve = (1 + d["returns"]).cumprod()
ma_tot = ma_curve.iloc[-1] - 1
bh_tot = bh_curve.iloc[-1] - 1
print("MA(10/30) crossover total return: %+.1f%%" % (100*ma_tot))
print("Buy & hold total return:          %+.1f%%" % (100*bh_tot))
print("-> the simple rule trails buy-and-hold" if ma_tot < bh_tot else "-> rule beat B&H this sample")

**Equity curves.** The crossover is smooth in some regimes but takes brutal hits when volatility spikes (crashes) — exactly when a static rule can't adapt. This is the pain that motivates a statistical model.

In [ ]:
plt.figure(figsize=(11,4))
plt.plot(ma_curve.index, ma_curve, label="MA(10/30) crossover", color="#db2777")
plt.plot(bh_curve.index, bh_curve, label="Buy & hold", color="#6b7280")
plt.axhline(1, color="k", lw=0.7); plt.legend(); plt.title("Simple rule vs buy & hold (growth of 1)")
plt.tight_layout(); plt.show()

## 5. The first time-series model: AutoRegressive AR(1)
A trader's hunch — *"if it went up, it'll keep going up"* — made **systematic**. An **AR(1)** model regresses
a variable on **its own previous value**:

$$ r_t = \alpha + \beta\, r_{t-1} + \varepsilon_t $$

- $\alpha$ — average return when there's no momentum.
- $\beta$ — how much yesterday's return carries into today.
  - $0 < \beta < 1$ → **trending** (positive follows positive).
  - $-1 < \beta < 0$ → **mean-reverting** (positive follows negative).
  - $\beta \approx 0$ → yesterday tells you **nothing**.

**AR(p)** generalises to the previous *p* values. We fit it on **returns**, not prices — and the next section
explains why.

## 6. Down-sample to weekly (our PayPal stand-in)
Lower-frequency strategies often use **weekly** bars. Resample OHLC with the right rule per column:
**Open = first**, **High = max**, **Low = min**, **Close = last**.

In [ ]:
wk = df.resample("W").agg({"Open":"first","High":"max","Low":"min","Close":"last"}).dropna()
wk["returns"] = wk["Close"].pct_change()
wk = wk.dropna()
print("weekly observations:", len(wk))
wk.tail()

## 7. Stationarity = stability
A series is **stationary** if its **mean** and **standard deviation** stay roughly constant through time (and
its autocorrelation structure doesn't drift). Rule of thumb:
- **Prices are non-stationary** — they trend; the mean in 2018 ≠ the mean in 2024.
- **Returns are (almost always) stationary** — they fluctuate around a stable ~0 mean.

That's *why* we model returns (or *differenced* prices): the statistics we rely on only behave when the
process is stable.

In [ ]:
roll = wk["returns"].rolling(26)
fig, ax = plt.subplots(figsize=(11,4))
ax.plot(wk.index, wk["returns"], color="#c084fc", lw=0.7, label="weekly returns")
ax.plot(wk.index, roll.mean(), color="#dc2626", lw=1.6, label="rolling mean (26w)")
ax.plot(wk.index, roll.std(), color="#16a34a", lw=1.6, label="rolling std (26w)")
ax.axhline(0, color="k", lw=0.7); ax.legend(); ax.set_title("Rolling mean stable, rolling vol wanders")
plt.tight_layout(); plt.show()

## 8. The ADF test — confirming stationarity statistically
The **Augmented Dickey-Fuller** test checks for a *unit root*. Read only the **p-value**:
- **p < 0.05** → reject the null → series is **stationary**.
- **p ≥ 0.05** → cannot reject → **non-stationary**.

(Null hypothesis = *non-stationary*.) We expect **prices: non-stationary**, **returns: stationary**.

In [ ]:
def adf(series, name):
    stat, p = adfuller(series.dropna())[:2]
    verdict = "STATIONARY" if p < 0.05 else "non-stationary"
    print(f"{name:18s}  ADF stat={stat:8.3f}  p={p:.4f}  -> {verdict}")

adf(wk["Close"],   "Weekly prices")
adf(wk["returns"], "Weekly returns")

## 9. ACF & PACF — how returns relate to their own past
- **ACF** (AutoCorrelation Function) at lag *k*: correlation of $r_t$ with $r_{t-k}$ — counting both **direct
  and indirect** links.
- **PACF** (Partial ACF): the **direct** correlation only, stripping out the chain through intermediate lags.

Bars poking **outside the blue band** (the *significance shadow*, $\pm 1.96/\sqrt{N}$) signal genuine
autocorrelation worth modelling. For nearly-efficient return series most bars sit *inside* the band.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
plot_acf(wk["returns"], lags=15, ax=ax[0]); ax[0].set_title("ACF — weekly returns")
plot_pacf(wk["returns"], lags=15, ax=ax[1], method="ywm"); ax[1].set_title("PACF — weekly returns")
plt.tight_layout(); plt.show()
band = 1.96/np.sqrt(len(wk))
print("significance band = +/- %.4f" % band)

## 10. A rolling AR(1) strategy
Professional systematic trading is **dynamic**: re-fit the model every period on a moving window and forecast
**one step ahead**. Here:
- window = **100 weeks**, model = `ARIMA(order=(1,0,0))` (an AR(1)),
- forecast next week's return; go **long** if forecast > 0, else **short**,
- step forward one week and repeat (a true **rolling forecast**, no look-ahead).

In [ ]:
r = wk["returns"].values
window = 100
sig = np.full(len(r), np.nan)

for t in range(window, len(r)):
    train = r[t-window:t]
    try:
        fit = ARIMA(train, order=(1,0,0)).fit()
        pred = fit.forecast(1)[0]
    except Exception:
        pred = 0.0
    sig[t] = 1.0 if pred > 0 else -1.0

wk2 = wk.copy()
wk2["sig"] = sig
wk2 = wk2.dropna()
wk2["strat"] = wk2["sig"] * wk2["returns"]      # signal already aligns to NEXT week's return
print("forecasts made:", len(wk2), "| long weeks:", int((wk2['sig']==1).sum()),
      "| short weeks:", int((wk2['sig']==-1).sum()))

## 11. Performance metrics — the honest scorecard
Total return, annualised **Sharpe** (×√52 for weekly), **win rate**, and **max drawdown**.

In [ ]:
def stats(ret, ann=52):
    cur = (1+ret).cumprod()
    tot = cur.iloc[-1]-1
    sharpe = (ret.mean()/ret.std())*np.sqrt(ann)
    dd = (cur/cur.cummax()-1).min()
    return tot, sharpe, dd, cur

s_tot, s_sh, s_dd, s_cur = stats(wk2["strat"])
b_tot, b_sh, b_dd, b_cur = stats(wk2["returns"])
win = (wk2["strat"] > 0).mean()

print("                    AR(1) rolling     Buy & hold")
print("Total return        %+9.1f%%     %+9.1f%%" % (100*s_tot, 100*b_tot))
print("Sharpe (annual)     %9.2f      %9.2f" % (s_sh, b_sh))
print("Max drawdown        %+9.1f%%     %+9.1f%%" % (100*s_dd, 100*b_dd))
print("Win rate            %9.1f%%" % (100*win))

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(s_cur.index, s_cur, label="AR(1) rolling", color="#db2777")
ax[0].plot(b_cur.index, b_cur, label="Buy & hold", color="#6b7280")
ax[0].axhline(1, color="k", lw=0.7); ax[0].legend(); ax[0].set_title("Rolling AR(1) vs buy & hold (growth of 1)")
dd = s_cur/s_cur.cummax()-1
ax[1].fill_between(dd.index, dd*100, 0, color="#f43f5e", alpha=.5)
ax[1].set_title("AR(1) strategy drawdown (%)")
plt.tight_layout(); plt.show()

### Reading the result honestly
On this data the rolling AR(1) was **right 54.4% of the time** — better than a coin flip — yet it still
**lost money** (**-18.5%** vs buy-and-hold's **+21.3%**). That is the classic trap: *right more than half the
time but losing on **asymmetric payoffs*** (the losing weeks were bigger than the winning ones). The instructor
saw the *same* model **outperform** on one data pull and **underperform** on another — the result is
**unstable**. Verdict: the logic is sound and the engine is professional, **but AR(1) alone — a single lag —
is too simple for production.** It assumes the next period behaves like the previous 100, which breaks at
regime shifts.

## 12. ARIMA = AutoRegressive + Integrated + Moving-average-of-errors
ARIMA(**p, d, q**) bolts three ideas together:

| Letter | Name | What it adds |
|---|---|---|
| **AR(p)** | AutoRegressive | uses the last *p* values of the series |
| **I(d)** | Integrated | **differences** the series *d* times to make it stationary (prices → returns is `d=1`-like) |
| **MA(q)** | Moving-average **of errors** | models the last *q* **forecast errors** (predicted − observed) to self-correct |

> ⚠️ This **MA** has **nothing** to do with the SMA/LMA price moving-averages from the crossover. Here "moving
> average" means the moving average of the model's **errors**.

**The exam analogy.** If a student *systematically* under-predicts their score by ~2 marks every week, you can
use that recurring error to correct the next prediction. ARIMA's MA term does exactly this for returns: it
learns from the pattern in past mistakes. Choosing the order **(p, d, q)** — guided by ADF (for *d*) and
ACF/PACF (for *p*, *q*) — is the craft we build on next.

## 13. Composite figure for the study pack

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 7))
ax[0,0].plot(ma_curve.index, ma_curve, color="#db2777", label="MA(10/30)")
ax[0,0].plot(bh_curve.index, bh_curve, color="#6b7280", label="Buy & hold")
ax[0,0].axhline(1, color="k", lw=.6); ax[0,0].legend(fontsize=8)
ax[0,0].set_title("1. Simple MA rule trails B&H", fontsize=10)

plot_acf(wk["returns"], lags=15, ax=ax[0,1]); ax[0,1].set_title("2. ACF of weekly returns", fontsize=10)

ax[1,0].plot(s_cur.index, s_cur, color="#db2777", label="AR(1) rolling")
ax[1,0].plot(b_cur.index, b_cur, color="#6b7280", label="Buy & hold")
ax[1,0].axhline(1, color="k", lw=.6); ax[1,0].legend(fontsize=8)
ax[1,0].set_title("3. Rolling AR(1) vs B&H", fontsize=10)

ax[1,1].fill_between(dd.index, dd*100, 0, color="#f43f5e", alpha=.5)
ax[1,1].set_title("4. AR(1) drawdown (deep!)", fontsize=10)
plt.tight_layout(); plt.savefig("chart_1_arima.png", dpi=110, bbox_inches="tight"); plt.show()
print("saved chart_1_arima.png")

## Takeaways
1. **Simple MA crossovers fail at regime shifts** — they can't adapt; here the rule trailed buy-and-hold.
2. **AR(1)**: $r_t = \alpha + \beta r_{t-1} + \varepsilon_t$ — systematise the momentum hunch; $\beta$ sign
   says trending vs mean-reverting.
3. **Model returns, not prices** — returns are **stationary** (confirmed by **ADF p < 0.05**); prices aren't.
4. **ACF/PACF** reveal which lags carry real (direct vs indirect) autocorrelation; bars outside the
   $\pm1.96/\sqrt N$ band matter.
5. **Rolling AR(1)** re-fits every week and forecasts one step ahead — professional and adaptive, but a single
   lag is **too simple**: barely beats a coin flip and the **drawdown is brutal**.
6. **ARIMA(p,d,q)** = AR (past values) + I (differencing for stationarity) + MA (learning from past **errors**)
   — the foundation for the order-selection craft in Part 2.

# Additive validation layer

The cells below are appended only in this validated copy.

In [ ]:
import pandas as pd
from pathlib import Path
base = Path('.')
files = ['asq01_html_audit.csv', 'asq01_source_inventory.csv', 'asq01_dependency_execution.csv', 'asq01_ma_weekly_metrics.csv', 'asq01_acf_stationarity_checks.csv', 'asq01_rolling_ar_metrics.csv', 'asq01_validation_controls.csv']
data = {f: pd.read_csv(base / f) for f in files}
{k: v.shape for k, v in data.items()}

## 1. Dependency and source audit

In [ ]:
print(data['asq01_source_inventory.csv'].to_string(index=False))
print(data['asq01_dependency_execution.csv'].to_string(index=False))
dep = data['asq01_dependency_execution.csv'].set_index('metric')
assert dep.loc['statsmodels_available','value'] in ['yes','no']

## 2. Market and stationarity metrics

In [ ]:
print(data['asq01_ma_weekly_metrics.csv'].to_string(index=False))
print(data['asq01_acf_stationarity_checks.csv'].to_string(index=False))
ma = data['asq01_ma_weekly_metrics.csv'].set_index('metric')
assert int(ma.loc['weekly_observations','value']) == 203

## 3. Rolling AR and controls

In [ ]:
print(data['asq01_rolling_ar_metrics.csv'].to_string(index=False))
print(data['asq01_validation_controls.csv'].to_string(index=False))
ar = data['asq01_rolling_ar_metrics.csv'].set_index('metric')
assert int(ar.loc['forecast_rows','value']) == 103